# Transportation Telemetry Predictive Timing Demo

## Project Question

**Can a real-time transportation telemetry platform predict train delay timing and support predictive maintenance decision-making using sensor, location, and operational event data?**

This notebook demonstrates:
1. Synthetic train telemetry generation
2. Bronze / Silver / Gold data engineering pipeline
3. Neural network predictive timing model
4. Delay risk classification and delay-minute regression
5. Evaluation metrics and visual outputs
6. Architecture interpretation for a Sr/Staff Data Engineer portfolio


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

print("Project root:", ROOT)

## 1. Generate synthetic telemetry data

In [ ]:
from src.data_generator import generate_synthetic_telemetry

df = generate_synthetic_telemetry(
    n_events=5000,
    output_path=ROOT / "data/raw/train_telemetry_events.csv"
)

display(df.head())
print(df.shape)

## 2. Run Bronze / Silver / Gold pipeline

In [ ]:
from src.pipeline import bronze_ingest, silver_clean, gold_features

bronze_df = bronze_ingest(
    ROOT / "data/raw/train_telemetry_events.csv",
    ROOT / "data/bronze/telemetry_bronze.parquet"
)

silver_df = silver_clean(
    ROOT / "data/bronze/telemetry_bronze.parquet",
    ROOT / "data/silver/telemetry_silver.parquet"
)

gold_df = gold_features(
    ROOT / "data/silver/telemetry_silver.parquet",
    ROOT / "data/gold/train_delay_features.parquet"
)

display(gold_df.head())
print(gold_df.shape)

## 3. Train neural network predictive timing model

In [ ]:
from src.model import train_model

metrics = train_model(
    gold_path=ROOT / "data/gold/train_delay_features.parquet",
    model_path=ROOT / "outputs/models/delay_timing_nn.pt",
    scaler_path=ROOT / "outputs/models/scaler.joblib",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    report_path=ROOT / "outputs/tables/classification_report.csv",
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    epochs=30,
)

metrics

## 4. Generate visual outputs

In [ ]:
from src.visualization import generate_figures

figures = generate_figures(
    predictions_path=ROOT / "outputs/tables/predictions.csv",
    loss_path=ROOT / "outputs/tables/training_loss.csv",
    metrics_path=ROOT / "outputs/tables/model_metrics.json",
    output_dir=ROOT / "outputs/figures",
)

figures

## 5. Review prediction outputs

In [ ]:
import pandas as pd

pred = pd.read_csv(ROOT / "outputs/tables/predictions.csv")
report = pd.read_csv(ROOT / "outputs/tables/classification_report.csv")

display(pred.head())
display(report)

## Final Interpretation

This project shows a local, runnable version of a transportation telemetry platform. It demonstrates how real-time sensor events can move through a medallion lakehouse pipeline and feed a neural network model that predicts delay risk and delay minutes.

This is a synthetic portfolio project. In production, it would connect to Kafka/Kinesis, Spark/Flink, Delta/Iceberg, cloud storage, orchestration, monitoring, and secure enterprise governance.
